In [2]:
import pandas as pd
from datasets import Dataset


train_df = pd.read_csv("bias_train.csv")
test_df = pd.read_csv("bias_test.csv")

train_df.head()

c:\Users\Asus\Documents\GitHub\byee\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,text,sexism_label,racism_label
0,"Then, she's a keeper. 😉",0,-1
1,This is like the Metallica video where the poo...,0,-1
2,woman?,0,-1
3,Unlicensed day care worker reportedly tells co...,0,-1
4,[USER] Leg day is easy. Hot girls who wear min...,1,-1


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

In [5]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )


train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map: 100%|██████████| 4400/4400 [00:00<00:00, 14285.71 examples/s]


In [9]:
train_dataset 

Dataset({
    features: ['text', 'sexism_label', 'racism_label', 'input_ids', 'attention_mask'],
    num_rows: 15599
})

In [10]:
train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "sexism_label",
        "racism_label"
    ]
)

test_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "sexism_label",
        "racism_label"
    ]
)

In [11]:
import torch
import torch.nn as nn
from transformers import AutoModel

class MultiTaskBERT(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = AutoModel.from_pretrained("roberta-base")

        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.3)

        self.sexism_head = nn.Linear(hidden_size, 2)
        self.racism_head = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, attention_mask):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled_output = outputs.last_hidden_state[:,0]

        pooled_output = self.dropout(pooled_output)

        sexism_logits = self.sexism_head(pooled_output)
        racism_logits = self.racism_head(pooled_output)

        return sexism_logits, racism_logits

In [12]:
model = MultiTaskBERT()

c:\Users\Asus\Documents\GitHub\byee\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 7575.11it/s]
RobertaModel LOAD REPORT from: roberta-base
Key      

In [13]:
loss_fn = nn.CrossEntropyLoss()


def compute_loss(sexism_logits, racism_logits, sexism_labels, racism_labels):

    loss = 0

    sexism_mask = sexism_labels != -1
    racism_mask = racism_labels != -1

    if sexism_mask.sum() > 0:
        loss += loss_fn(
            sexism_logits[sexism_mask],
            sexism_labels[sexism_mask]
        )

    if racism_mask.sum() > 0:
        loss += loss_fn(
            racism_logits[racism_mask],
            racism_labels[racism_mask]
        )

    return loss

In [14]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [16]:
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

epochs = 3

for epoch in range(epochs):

    model.train()

    total_loss = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in progress_bar:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        sexism_labels = batch["sexism_label"].to(device)
        racism_labels = batch["racism_label"].to(device)

        sexism_logits, racism_logits = model(input_ids, attention_mask)

        loss = compute_loss(
            sexism_logits,
            racism_logits,
            sexism_labels,
            racism_labels
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        progress_bar.set_postfix({
            "batch_loss": loss.item()
        })

    avg_loss = total_loss / len(train_loader)

    print(f"\nEpoch {epoch+1}/{epochs} finished")
    print(f"Average Loss: {avg_loss:.4f}")

Epoch 1/3: 100%|██████████| 975/975 [3:46:38<00:00, 13.95s/it, batch_loss=0.435]      



Epoch 1/3 finished
Average Loss: 0.6759


Epoch 2/3: 100%|██████████| 975/975 [48:25<00:00,  2.98s/it, batch_loss=0.406] 



Epoch 2/3 finished
Average Loss: 0.4988


Epoch 3/3: 100%|██████████| 975/975 [46:52<00:00,  2.89s/it, batch_loss=0.221] 


Epoch 3/3 finished
Average Loss: 0.3907


In [ ]:
text = "Women are not equal to men"

# tokenize
inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=256
)

# move to device
inputs = {k: v.to(device) for k, v in inputs.items()}

# inference
model.eval()
with torch.no_grad():
    sexism_logits, racism_logits = model(**inputs)

sexism_pred = torch.argmax(sexism_logits, dim=1)
racism_pred = torch.argmax(racism_logits, dim=1)

print("Sexism prediction:", sexism_pred.item())
print("Racism prediction:", racism_pred.item())

Sexism prediction: 0
Racism prediction: 0


In [18]:
import os

save_path = "bias_multitask_model"

os.makedirs(save_path, exist_ok=True)

torch.save(model, "bias_multitask_model/full_model.pt")